In [ ]:
!wget "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
!pip install flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

In [ ]:
import os
import numpy as np
import random

# 클라우드
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "patent_disc"
os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
os.environ["HF_HOME"] = "hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForMaskedLM, TrainingArguments, Trainer
from datasets import load_dataset

In [ ]:
# Config
config = {
    "seed": 42,
    "learning_rate": 5e-5,
    "epochs": 5,
    "weight_decay": 1e-5,
    "warmup_ratio": 0.06,
    "adam_beta2": 0.98,          # ModernBERT 사전학습 레시피
    "adam_epsilon": 1e-6,
    "model_name": "skt/A.X-Encoder-base",
    "max_len": 2048,
    "eff_batch": 128,            # probe 후 조정
    "micro_batch": 4,           # probe 후 조정
    "eval_micro_batch": 4,      # probe 후 조정 (MLM eval은 vocab 로짓이 커 작게)
    "repo_train": "ingyoun/A.X-patent-tapt-mlm",
    "rev": "9708f9c404ace91efd25c06fac2d73413616f4ef",
}

config["grad_accum"] = config["eff_batch"] // config["micro_batch"]   # micro×accum = eff_batch
config["run_name"] = "axenc-tapt-mlm"

os.environ["WANDB_NOTEBOOK_NAME"] = os.path.abspath("13_01_TAPT_MLM.ipynb")

In [ ]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

In [ ]:
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

cuda


## 데이터셋

In [ ]:
dataset = load_dataset(
    "ingyoun/patent-clean-text-modernbert-tokenized",
    cache_dir="/content/hf_cache",
)

dataset

README.md:   0%|          | 0.00/679 [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/531M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/519M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/542M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/89.6M [00:00<?, ?B/s]

val-00000-of-00001.parquet:   0%|          | 0.00/87.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/201616 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11244 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/11132 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201616
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11244
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11132
    })
})

## Prep

In [ ]:
model = AutoModelForMaskedLM.from_pretrained(
        pretrained_model_name_or_path=config["model_name"],
        dtype=torch.float32,
        attn_implementation="flash_attention_2",            # len8192와 구현 일치
    ).to("cuda")                                            # probe가 GPU 메모리를 재려면 선 배치 필요

config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/299M [00:00<?, ?B/s]

[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertForMaskedLM is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`
[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`


Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], revision=config["rev"])
collator = DataCollatorForLanguageModeling(
    tokenizer, mlm=True, mlm_probability=0.30,
)

tokenizer_config.json:   0%|          | 0.00/6.95k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

In [ ]:
# max_len 청킹 + MLM에 불필요한 컬럼 제거 (input_ids만 남긴다)
# 저장된 input_ids는 truncation 없이 최대 10,523 토큰. 절단은 >2048 문서(3.67%)의 꼬리를 버려 총 토큰의 5.87%(약 9.4M)를 잃는다.
# 문서를 max_len 창으로 비겹침 분할해 전체 토큰을 학습에 넣는다.
# 각 청크는 <s>…</s>로 재구성(사전학습 포맷 일치) — 짧은 문서(≤max_len)는 원본과 동일.
# labels(다중핫)·document_id(문자열) 등 제거.
BOS, EOS = tokenizer.bos_token_id, tokenizer.eos_token_id   # <s>=0, </s>=1
STRIDE = config["max_len"] - 2                              # 특수토큰 2칸 확보

def _chunk(batch):
    out = []
    for x in batch["input_ids"]:
        core = x[1:-1] if len(x) >= 2 and x[0] == BOS and x[-1] == EOS else x
        if not core:
            out.append([BOS, EOS])
            continue
        for i in range(0, len(core), STRIDE):
            out.append([BOS] + core[i:i + STRIDE] + [EOS])
    return {"input_ids": out}

_drop = [c for c in dataset["train"].column_names if c != "input_ids"]
dataset = dataset.map(_chunk, batched=True, remove_columns=_drop)
dataset

Map:   0%|          | 0/201616 [00:00<?, ? examples/s]

Map:   0%|          | 0/11244 [00:00<?, ? examples/s]

Map:   0%|          | 0/11132 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 211159
    })
    test: Dataset({
        features: ['input_ids'],
        num_rows: 11807
    })
    val: Dataset({
        features: ['input_ids'],
        num_rows: 11658
    })
})

## 훈련

In [ ]:
## micro batch 확인 — worst-case 배치로 실제 스텝을 돌려 peak 측정
def probe_batches(model, vocab_size, seq_len, train_mb=(1, 2, 4), eval_mb=(2, 4, 8)):
    """
    worst case = 모든 시퀀스가 seq_len이고 mask가 전부 1(패딩 0).

    group_by_length가 시퀀스가 가장 큰 배치를 만들고 초반 할당한다.
    (random 샘플링이어도 배치에 장문 하나만 섞이면 배치 전체가 그 길이로 패딩된다).

    - AdamW state를 상주시킨 **정상상태** peak를 잰다.
      lr=0이라 가중치는 변하지 않는다 → 사전학습 가중치를 훼손하지 않음.
    - MLM 손실(labels=input_ids)로 forward — 50k vocab 로짓이 MLM 메모리를 지배하므로 실제 경로와 일치.
    - eval도 잰다: forward-only(no_grad)라 싸지만 eval 배치 역시 배치 내 최댓값으로 패딩되고,
      eval OOM은 첫 에폭 끝에서 런을 죽인다.
    """
    dev = model.device
    opt = torch.optim.AdamW(model.parameters(), lr=0.0)   # lr=0 → state만 할당, 가중치 불변

    def _mk(mb):
        ids = torch.randint(5, vocab_size, (mb, seq_len), device=dev)
        return ids, torch.ones_like(ids)

    model.train()
    for mb in train_mb:
        try:
            ids, mask = _mk(mb)
            for step in (0, 1):                       # step0: AdamW state 할당 / step1: 정상상태 peak 측정
                if step == 1:
                    torch.cuda.reset_peak_memory_stats()
                with torch.autocast("cuda", dtype=torch.bfloat16):   # Trainer(bf16=True)와 동일 경로
                    out = model(input_ids=ids, attention_mask=mask, labels=ids)
                    loss = out.loss
                loss.backward()                                      # backward는 autocast 밖
                opt.step()
                opt.zero_grad(set_to_none=True)
            print(f"train micro={mb:>3}: peak {torch.cuda.max_memory_allocated()/1e9:5.1f} GB  OK")
        except torch.cuda.OutOfMemoryError:
            print(f"train micro={mb:>3}: OOM")
            opt.zero_grad(set_to_none=True)
            break                                     # 이후 후보는 자명하게 OOM
        finally:
            torch.cuda.empty_cache()

    model.eval()
    for mb in eval_mb:                                # 옵티마이저 state가 상주한 실제 조건에서 측정
        try:
            torch.cuda.reset_peak_memory_stats()
            ids, mask = _mk(mb)
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                model(input_ids=ids, attention_mask=mask, labels=ids)
            print(f"eval  micro={mb:>3}: peak {torch.cuda.max_memory_allocated()/1e9:5.1f} GB  OK")
        except torch.cuda.OutOfMemoryError:
            print(f"eval  micro={mb:>3}: OOM")
            break
        finally:
            torch.cuda.empty_cache()

    del opt                                           # Trainer가 자체 옵티마이저를 새로 만들도록 정리
    model.zero_grad(set_to_none=True)
    model.train()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()              # train() peak를 깨끗하게 재측정하기 위함


# 설정값 검증
probe_batches(
    model, tokenizer.vocab_size, config["max_len"],
    train_mb=(4, 8, 16, 32, 64, 96, 128, 160, 192),
    eval_mb=(4, 8, 16, 32, 64, 96, 128, 256, 512),
)

train micro=  4: peak  12.9 GB  OK
train micro=  8: OOM
eval  micro=  4: peak  21.6 GB  OK
eval  micro=  8: OOM


In [ ]:
import gc

# probe_batches가 옵티마이저·캐시를 내부에서 정리하므로 여기선 GC만 한 번 더 돌린다
gc.collect()
torch.cuda.empty_cache()
print(f"probe 후 잔여 allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB (모델 가중치)")

probe 후 잔여 allocated: 0.6 GB (모델 가중치)


In [ ]:
training_args = TrainingArguments(
    output_dir='/content/results',
    seed=config["seed"],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
    lr_scheduler_type="linear",
    warmup_ratio=config["warmup_ratio"],
    adam_beta2=config["adam_beta2"],                    # ModernBERT 사전학습 레시피
    adam_epsilon=config["adam_epsilon"],
    per_device_train_batch_size=config["micro_batch"],
    per_device_eval_batch_size=config["eval_micro_batch"],
    gradient_accumulation_steps=config["grad_accum"],
    train_sampling_strategy="group_by_length",          # 유사 길이 배치
    num_train_epochs=config["epochs"],
    bf16=True,                                          # flash-attention-2 반정밀도 경로
    eval_strategy="epoch",                              # 에폭별 MLM val loss (과적합 가드)
    save_strategy="epoch",                              # 에폭별 체크포인트
    save_total_limit=None,                              # 5에폭 전부 보존
    logging_steps=50,
    report_to="wandb",
    run_name=config["run_name"],
    push_to_hub=True,
    hub_model_id=config["repo_train"],
    hub_strategy="all_checkpoints",
    hub_always_push=True,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=collator,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
from huggingface_hub import snapshot_download
import shutil

def copy_checkpoint(repo_train, ckpt_name, dst):
    src_root = snapshot_download(repo_id=repo_train, allow_patterns=f"{ckpt_name}/*")
    shutil.copytree(os.path.join(src_root, ckpt_name), dst, dirs_exist_ok=True)

copy_checkpoint(config["repo_train"], "checkpoint-3300", "/content/results/checkpoint-3300")

In [ ]:
trainer.train(resume_from_checkpoint="/content/results/checkpoint-3300")

[transformers] There were missing keys in the checkpoint model loaded: ['decoder.weight'].
wandb: WARNING WANDB_NOTEBOOK_NAME should be a path to a notebook file, couldn't find /content/13_01_TAPT_MLM.ipynb.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
3,0.389013,0.391717
4,0.375958,0.379318
5,0.371327,0.374164


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8250, training_loss=0.2277274063572739, metrics={'train_runtime': 30700.1116, 'train_samples_per_second': 34.391, 'train_steps_per_second': 0.269, 'total_flos': 5.343598658970706e+17, 'train_loss': 0.2277274063572739, 'epoch': 5.0})

|Epoch|Training Loss|Validation Loss|
|---|---|----|
|1|0.433938|0.427220|
|2|0.406178|0.404192